#### Retail Orders Analysis

- Data Loading and Inspection


In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ['KAGGLE_USERNAME'] = os.getenv('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = os.getenv('KAGGLE_KEY')

In [3]:
!kaggle datasets download -d ankitbansal06/retail-orders

Dataset URL: https://www.kaggle.com/datasets/ankitbansal06/retail-orders
License(s): CC0-1.0
  0%|                                                | 0.00/200k [00:00<?, ?B/s]
100%|████████████████████████████████████████| 200k/200k [00:00<00:00, 81.9MB/s]


In [4]:
import zipfile

with zipfile.ZipFile('retail-orders.zip', 'r') as zip_ref:
    zip_ref.extractall('orders')

In [5]:
import pandas as pd

retail = pd.read_csv('orders/orders.csv')
retail.head(30)

,Order Id,Order Date,Ship Mode,Segment,Country,City,State,Postal Code,Region,Category,Sub Category,Product Id,cost price,List Price,Quantity,Discount Percent
0,1,2023-03-01,Second Class,Consumer,United States,Henderson,Kentucky,42420,South,Furniture,Bookcases,FUR-BO-10001798,240,260,2,2
1,2,2023-08-15,Second Class,Consumer,United States,Henderson,Kentucky,42420,South,Furniture,Chairs,FUR-CH-10000454,600,730,3,3
2,3,2023-01-10,Second Class,Corporate,United States,Los Angeles,California,90036,West,Office Supplies,Labels,OFF-LA-10000240,10,10,2,5
3,4,2022-06-18,Standard Class,Consumer,United States,Fort Lauderdale,Florida,33311,South,Furniture,Tables,FUR-TA-10000577,780,960,5,2
4,5,2022-07-13,Standard Class,Consumer,United States,Fort Lauderdale,Florida,33311,South,Office Supplies,Storage,OFF-ST-10000760,20,20,2,5
5,6,2022-03-13,Not Available,Consumer,United States,Los Angeles,California,90032,West,Furniture,Furnishings,FUR-FU-10001487,50,50,7,3
6,7,2022-12-28,Standard Class,Consumer,United States,Los Angeles,California,90032,West,Office Supplies,Art,OFF-AR-10002833,10,10,4,3
7,8,2022-01-25,Standard Class,Consumer,United States,Los Angeles,California,90032,West,Technology,Phones,TEC-PH-10002275,860,910,6,5
8,9,2023-03-23,Not Available,Consumer,United States,Los Angeles,California,90032,West,Office Supplies,Binders,OFF-BI-10003910,20,20,3,2
9,10,2023-05-16,Standard Class,Consumer,United States,Los Angeles,California,90032,West,Office Supplies,Appliances,OFF-AP-10002892,90,110,5,3


In [6]:
retail.shape

(9994, 16)

In [8]:
retail.isnull().sum()

Order Id            0
Order Date          0
Ship Mode           1
Segment             0
Country             0
City                0
State               0
Postal Code         0
Region              0
Category            0
Sub Category        0
Product Id          0
cost price          0
List Price          0
Quantity            0
Discount Percent    0
dtype: int64

In [9]:
retail.dtypes


Order Id             int64
Order Date          object
Ship Mode           object
Segment             object
Country             object
City                object
State               object
Postal Code          int64
Region              object
Category            object
Sub Category        object
Product Id          object
cost price           int64
List Price           int64
Quantity             int64
Discount Percent     int64
dtype: object

In [10]:
for col in retail.select_dtypes(include = 'object').columns:
    if col != 'Order Date':
        print(f"Unique values in '{col}': ")
        print(retail[col].unique())
        print('-' * 20)

Unique values in 'Ship Mode': 
['Second Class' 'Standard Class' 'Not Available' 'unknown' 'First Class'
 nan 'Same Day']
--------------------
Unique values in 'Segment': 
['Consumer' 'Corporate' 'Home Office']
--------------------
Unique values in 'Country': 
['United States']
--------------------
Unique values in 'City': 
['Henderson' 'Los Angeles' 'Fort Lauderdale' 'Concord' 'Seattle'
 'Fort Worth' 'Madison' 'West Jordan' 'San Francisco' 'Fremont'
 'Philadelphia' 'Orem' 'Houston' 'Richardson' 'Naperville' 'Melbourne'
 'Eagan' 'Westland' 'Dover' 'New Albany' 'New York City' 'Troy' 'Chicago'
 'Gilbert' 'Springfield' 'Jackson' 'Memphis' 'Decatur' 'Durham' 'Columbia'
 'Rochester' 'Minneapolis' 'Portland' 'Saint Paul' 'Aurora' 'Charlotte'
 'Orland Park' 'Urbandale' 'Columbus' 'Bristol' 'Wilmington' 'Bloomington'
 'Phoenix' 'Roseville' 'Independence' 'Pasadena' 'Newark' 'Franklin'
 'Scottsdale' 'San Jose' 'Edmond' 'Carlsbad' 'San Antonio' 'Monroe'
 'Fairfield' 'Grand Prairie' 'Redlands' 'H

In [ ]:
# # Replace entries 'Not Available' and 'unknown'  in 'Ship Mode' with NaN
import numpy as np

retail['Ship Mode'] = retail['Ship Mode'].replace(
    ['Not Available', 'unknown'], 
    np.nan)

print(retail['Ship Mode'].head(30))

0       Second Class
1       Second Class
2       Second Class
3     Standard Class
4     Standard Class
5                NaN
6     Standard Class
7     Standard Class
8                NaN
9     Standard Class
10               NaN
11               NaN
12    Standard Class
13    Standard Class
14               NaN
15    Standard Class
16    Standard Class
17      Second Class
18      Second Class
19      Second Class
20      Second Class
21    Standard Class
22    Standard Class
23      Second Class
24    Standard Class
25      Second Class
26      Second Class
27    Standard Class
28    Standard Class
29    Standard Class
Name: Ship Mode, dtype: object


In [16]:
# rename all DataFrame columns by lowercasing and replacing spaces with underscores.

retail.columns = retail.columns.str.lower().str.replace(' ', '_')

print(retail.columns)

Index(['order_id', 'order_date', 'ship_mode', 'segment', 'country', 'city',
       'state', 'postal_code', 'region', 'category', 'sub_category',
       'product_id', 'cost_price', 'list_price', 'quantity',
       'discount_percent'],
      dtype='object')


In [20]:
# 1. Compute 'discount' = List_Price * (Discount_Percent / 100)
retail['discount'] = retail['list_price'] * (retail['discount_percent'] / 100)
# 2. Compute 'selling_price' = List_Price - discount
retail['selling_price'] = retail['list_price'] - retail['discount']

# 3. Compute 'profit' = selling_price - cost_price
retail['profit'] = retail['selling_price'] - retail['cost_price']

# Verify the new columns
print(retail[['list_price', 'discount_percent', 'discount', 'selling_price', 'profit']].head(30))

    list_price  discount_percent  discount  selling_price  profit
0          260                 2       5.2          254.8    14.8
1          730                 3      21.9          708.1   108.1
2           10                 5       0.5            9.5    -0.5
3          960                 2      19.2          940.8   160.8
4           20                 5       1.0           19.0    -1.0
5           50                 3       1.5           48.5    -1.5
6           10                 3       0.3            9.7    -0.3
7          910                 5      45.5          864.5     4.5
8           20                 2       0.4           19.6    -0.4
9          110                 3       3.3          106.7    16.7
10        1710                 3      51.3         1658.7   188.7
11         910                 3      27.3          882.7   132.7
12          20                 3       0.6           19.4    -0.6
13         410                 2       8.2          401.8    41.8
14        

In [21]:
# Drop the specified columns
retail = retail.drop(columns=['cost_price', 'list_price', 'discount_percent'])

# Verify the remaining columns
print(retail.columns)

Index(['order_id', 'order_date', 'ship_mode', 'segment', 'country', 'city',
       'state', 'postal_code', 'region', 'category', 'sub_category',
       'product_id', 'quantity', 'discount', 'selling_price', 'profit'],
      dtype='object')


In [22]:
import pandas as pd

# Convert 'Order_Date' to datetime (ISO 8601 format: YYYY-MM-DD)
retail['order_date'] = pd.to_datetime(retail['order_date'], format='ISO8601')

# Verify conversion
print(retail['order_date'].head(30))

0    2023-03-01
1    2023-08-15
2    2023-01-10
3    2022-06-18
4    2022-07-13
5    2022-03-13
6    2022-12-28
7    2022-01-25
8    2023-03-23
9    2023-05-16
10   2023-03-31
11   2023-12-25
12   2022-02-11
13   2023-07-18
14   2023-11-09
15   2022-06-18
16   2022-02-04
17   2023-08-04
18   2022-01-23
19   2022-01-11
20   2022-10-05
21   2023-07-16
22   2023-05-06
23   2023-05-21
24   2023-02-24
25   2022-06-20
26   2022-02-08
27   2023-12-11
28   2022-08-21
29   2022-08-14
Name: order_date, dtype: datetime64[ns]
